In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

# Googleドライブの空き容量を確認（A100用の30Bクラスモデルは1つ約18〜20GB）
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
free_gb = free / (1024**3)
print(f"💾 Googleドライブ空き容量: {free_gb:.1f} GB")

# 空き容量が25GB以上あればドライブに永続保存、足りなければColab本体の高速SSDに保存
if free_gb >= 25:
    os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'
    print("✅ モデルはGoogleドライブ (/content/drive/MyDrive/ollama_models) に永続保存されます。")
else:
    os.environ['OLLAMA_MODELS'] = '/root/.ollama/models'
    print("⚠️ ドライブの空きが25GB未満のため、Colab本体のディスクにモデルを保存します。")


In [ ]:
import os
import subprocess
import time
import shutil
import re

# 破損ファイルや過去のゴミをクリーンアップ
for path in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/content/cloudflared.deb"]:
    if os.path.exists(path):
        try:
            if os.path.isdir(path):
                shutil.rmtree(path)
            else:
                os.remove(path)
        except Exception:
            pass

def download_file(url, filepath):
    try:
        subprocess.run(["curl", "-L", "-o", filepath, url], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

# 0. 依存パッケージ (zstd + A100 GPU自動検出用pciutils) をインストール
print("📦 依存パッケージ (zstd, pciutils) をインストール中...")
subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "zstd", "pciutils"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 1. Ollama のインストール (A100 Ampereアーキテクチャを自動認識)
print("📦 Ollama をインストール中...")
!curl -fsSL https://ollama.com/install.sh | sh

# 2. cloudflared のインストール
print("📦 Cloudflare Tunnel を準備中...")
cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
success_cf = download_file(cf_url, "/content/cloudflared.deb")

if success_cf and os.path.exists("/content/cloudflared.deb") and os.path.getsize("/content/cloudflared.deb") > 5000000:
    !dpkg -i /content/cloudflared.deb > /dev/null 2>&1
    print("✅ Cloudflare Tunnel の準備完了")
else:
    raise RuntimeError("❌ Cloudflare Tunnel のダウンロードに失敗しました。")

# 3. A100 GPU (40GB/80GB VRAM) 最適化パラメーターで Ollama サーバーを起動
print("🚀 Ollama サーバーを A100 最適化パラメーターで起動中...")
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '65536'     # ★A100の大容量VRAMを活かして64Kコンテキストに拡張
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'        # ★A100のFlash Attentionを有効化（長文の高速化＆省メモリ化）
os.environ['OLLAMA_KV_CACHE_TYPE'] = 'q8_0'       # ★KVキャッシュを8bit量子化（精度を保ったままVRAM消費を半減）
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'      # ★30B級モデルの切り替え時にVRAM溢れ(OOM)を防止
os.environ['OLLAMA_NUM_PARALLEL'] = '1'           # ★1セッションに全VRAM・計算力を集中
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'            # ★使用中のモデルをVRAMに常駐

subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

# 4. A100 GPU おすすめAIモデル（コーディング＆Googleモデル・ツール呼び出し対応）
MODELS = [
    "qwen3-coder:30b",    # ★本命1: エージェント・コーディング特化 (MoEで爆速＆ツール呼び出し最強)
    "gemma4:31b",         # ★本命2(Google): Google最新フラッグシップ31B (推論・コード・ツール対応)
    "gemma4:26b",         # ★Google高速版: Gemma 4 MoEモデル (26B/実効4Bで超高速レスポンス)
    "qwen3:32b",          # ★重厚長大: 32Bフルモデル (複雑な設計・論理思考・デバッグ向け)
    # "qwen2.5-coder:32b", # （任意）定番の32Bコーディング特化モデルを使いたい場合はコメント解除
    # "gemma3:27b",        # （任意）Google Gemma 3 27Bを使いたい場合はコメント解除
]

for model in MODELS:
    print(f"\n📥 モデル ({model}) を準備中...")
    !ollama pull {model}

# 5. Cloudflare Tunnel を起動してURLを発行
print("\n🌐 Cloudflare Tunnel を起動中...")
log_path = "/content/cloudflared.log"
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434",
     "--http-host-header", "localhost:11434"],
    stdout=open(log_path, "w"), stderr=subprocess.STDOUT
)

url = None
for _ in range(60):
    time.sleep(1)
    if os.path.exists(log_path):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            url = m.group(0)
            break

if url:
    print("\n" + "="*65)
    print("🎉 A100 準備完了！以下のURLを Zoo Code / Cline の Base URL に貼ってください：")
    print(f"👉 {url}")
    print("="*65)
    print("📌 使えるモデルID一覧（Zoo CodeのModel欄に入力）：")
    for m in MODELS:
        print(f"   • {m}")
    print("\n⚠️ このノートブックを閉じるとURLは無効になります。")
else:
    print("❌ URLを取得できませんでした。/content/cloudflared.log を確認してください。")
